In [1]:
import os
import random

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm


class FundusVesselExtractor:
    def __init__(
        self,
        input_green_dir="preprocessed_green_images",
        input_mask_dir="fundus_masks",
        output_mask_dir="vessel_masks",
        output_skeleton_dir="vessel_skeletons",
        debug_dir="vessel_debug_samples",
        metrics_csv="vessel_extraction_metrics.csv",
        debug_sample_count=1,
        random_seed=42,
        grid_size=8,

        process_scale=0.85,

        effective_mask_erode_size=20,

        frangi_sigmas=(0.8, 1.0, 1.4, 2.0, 2.8),
        frangi_beta=0.35,
        frangi_gamma_min=2.0,

        gabor_ksize=17,
        gabor_sigmas=(2.0, 3.0),
        gabor_lambdas=(8.0, 12.0),
        gabor_gamma=0.45,
        gabor_angle_step=30,

        frangi_weight=0.65,
        gabor_weight=0.35,
        use_product_response=True,

        response_floor_percentile=62,
        response_cap_percentile=99.5,
        response_median_ksize=3,

        high_percentile=87.0,
        low_ratio=0.38,
        min_valid_response=5,
        gap_close_size=5,

        weak_frangi_ratio=0.50,
        weak_frangi_min_response=4,
        weak_connection_dilate_size=11,

        min_component_area=20,
        min_component_length=10,
        min_response_mean=5.0,
        min_aspect_ratio=1.4,
        max_component_extent=0.92,

        min_linearity=0.12,
        connected_min_linearity_ratio=0.55,

        connected_area_ratio=0.45,
        connected_length_ratio=0.45,
        connected_response_ratio=0.55,
        strong_connection_dilate_size=7,

        remove_image_border_components=True,
        image_border_margin=4,

        use_skeleton=True,
        save_debug=True,
    ):
        self.input_green_dir = input_green_dir
        self.input_mask_dir = input_mask_dir
        self.output_mask_dir = output_mask_dir
        self.output_skeleton_dir = output_skeleton_dir
        self.debug_dir = debug_dir
        self.metrics_csv = metrics_csv
        self.debug_sample_count = debug_sample_count
        self.random_seed = random_seed
        self.grid_size = grid_size

        self.process_scale = process_scale

        self.effective_mask_erode_size = effective_mask_erode_size

        self.frangi_sigmas = frangi_sigmas
        self.frangi_beta = frangi_beta
        self.frangi_gamma_min = frangi_gamma_min

        self.gabor_ksize = gabor_ksize
        self.gabor_sigmas = gabor_sigmas
        self.gabor_lambdas = gabor_lambdas
        self.gabor_gamma = gabor_gamma
        self.gabor_angle_step = gabor_angle_step

        self.frangi_weight = frangi_weight
        self.gabor_weight = gabor_weight
        self.use_product_response = use_product_response

        self.response_floor_percentile = response_floor_percentile
        self.response_cap_percentile = response_cap_percentile
        self.response_median_ksize = response_median_ksize

        self.high_percentile = high_percentile
        self.low_ratio = low_ratio
        self.min_valid_response = min_valid_response
        self.gap_close_size = gap_close_size

        self.weak_frangi_ratio = weak_frangi_ratio
        self.weak_frangi_min_response = weak_frangi_min_response
        self.weak_connection_dilate_size = weak_connection_dilate_size

        self.min_component_area = min_component_area
        self.min_component_length = min_component_length
        self.min_response_mean = min_response_mean
        self.min_aspect_ratio = min_aspect_ratio
        self.max_component_extent = max_component_extent

        self.min_linearity = min_linearity
        self.connected_min_linearity_ratio = connected_min_linearity_ratio

        self.connected_area_ratio = connected_area_ratio
        self.connected_length_ratio = connected_length_ratio
        self.connected_response_ratio = connected_response_ratio
        self.strong_connection_dilate_size = strong_connection_dilate_size

        self.remove_image_border_components = remove_image_border_components
        self.image_border_margin = image_border_margin

        self.use_skeleton = use_skeleton
        self.save_debug = save_debug

        self.gabor_kernels = self.build_gabor_kernels()

        os.makedirs(self.output_mask_dir, exist_ok=True)
        os.makedirs(self.output_skeleton_dir, exist_ok=True)
        os.makedirs(self.debug_dir, exist_ok=True)

    def build_gabor_kernels(self):
        kernels = []

        angles = list(
            range(
                0,
                180,
                self.gabor_angle_step
            )
        )

        for sigma in self.gabor_sigmas:
            for lambd in self.gabor_lambdas:
                for angle in angles:
                    theta = np.deg2rad(
                        angle
                    )

                    kernel = cv2.getGaborKernel(
                        ksize=(self.gabor_ksize, self.gabor_ksize),
                        sigma=sigma,
                        theta=theta,
                        lambd=lambd,
                        gamma=self.gabor_gamma,
                        psi=0,
                        ktype=cv2.CV_32F
                    )

                    kernel = kernel - np.mean(
                        kernel
                    )

                    norm = np.sum(
                        np.abs(kernel)
                    )

                    if norm > 1e-6:
                        kernel = kernel / norm

                    kernels.append(
                        kernel
                    )

        return kernels

    def get_base_image_id(self, green_file):
        image_id = os.path.splitext(
            green_file
        )[0]

        base_id = image_id.replace(
            "_green_preprocessed",
            ""
        )

        return base_id, image_id

    def load_fundus_mask(self, green_file, green):
        base_id, _ = self.get_base_image_id(
            green_file
        )

        mask_file = base_id + "_fundus_mask.png"

        mask_path = os.path.join(
            self.input_mask_dir,
            mask_file
        )

        if os.path.exists(mask_path):
            fundus_mask = cv2.imread(
                mask_path,
                cv2.IMREAD_GRAYSCALE
            )

            if fundus_mask is not None:
                if fundus_mask.shape != green.shape:
                    fundus_mask = cv2.resize(
                        fundus_mask,
                        (green.shape[1], green.shape[0]),
                        interpolation=cv2.INTER_NEAREST
                    )

                fundus_mask = (fundus_mask > 0).astype(np.uint8) * 255

                return fundus_mask, mask_path, "saved_mask"

        fundus_mask = self.create_fundus_mask_from_green(
            green
        )

        return fundus_mask, "", "recreated_from_green"

    def create_fundus_mask_from_green(self, green):
        _, mask = cv2.threshold(
            green,
            5,
            255,
            cv2.THRESH_BINARY
        )

        kernel = np.ones(
            (21, 21),
            np.uint8
        )

        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            kernel
        )

        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        clean_mask = np.zeros_like(
            mask
        )

        if len(contours) > 0:
            largest_contour = max(
                contours,
                key=cv2.contourArea
            )

            cv2.drawContours(
                clean_mask,
                [largest_contour],
                -1,
                255,
                thickness=-1
            )
        else:
            clean_mask = mask.copy()

        clean_mask = cv2.erode(
            clean_mask,
            np.ones((7, 7), np.uint8),
            iterations=1
        )

        return clean_mask

    def resize_for_processing(self, green, fundus_mask):
        if self.process_scale >= 0.999:
            return green, fundus_mask

        new_w = max(
            1,
            int(green.shape[1] * self.process_scale)
        )

        new_h = max(
            1,
            int(green.shape[0] * self.process_scale)
        )

        green_small = cv2.resize(
            green,
            (new_w, new_h),
            interpolation=cv2.INTER_AREA
        )

        mask_small = cv2.resize(
            fundus_mask,
            (new_w, new_h),
            interpolation=cv2.INTER_NEAREST
        )

        mask_small = (mask_small > 0).astype(np.uint8) * 255

        return green_small, mask_small

    def resize_result_to_original(self, image, original_shape):
        if image.shape == original_shape:
            return image

        resized = cv2.resize(
            image,
            (original_shape[1], original_shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

        resized = (resized > 0).astype(np.uint8) * 255

        return resized

    def create_effective_mask(self, fundus_mask):
        if self.effective_mask_erode_size <= 1:
            effective_mask = fundus_mask.copy()
        else:
            kernel = np.ones(
                (
                    self.effective_mask_erode_size,
                    self.effective_mask_erode_size
                ),
                np.uint8
            )

            effective_mask = cv2.erode(
                fundus_mask,
                kernel,
                iterations=1
            )

        effective_mask = (effective_mask > 0).astype(np.uint8) * 255

        return effective_mask

    def normalize_inside_mask(self, image, mask):
        normalized = np.zeros_like(
            mask,
            dtype=np.uint8
        )

        image_f = image.astype(
            np.float32
        )

        pixels = image_f[mask > 0]

        if pixels.size == 0:
            return normalized

        p1 = np.percentile(
            pixels,
            1
        )

        p99 = np.percentile(
            pixels,
            99
        )

        if p99 <= p1:
            return normalized

        temp = (image_f - p1) / (p99 - p1)

        temp = np.clip(
            temp,
            0.0,
            1.0
        )

        temp = (temp * 255).astype(
            np.uint8
        )

        normalized = cv2.bitwise_and(
            temp,
            temp,
            mask=mask
        )

        return normalized

    def normalize_response_with_floor(
        self,
        response,
        mask,
        floor_percentile=None,
        cap_percentile=None
    ):
        if floor_percentile is None:
            floor_percentile = self.response_floor_percentile

        if cap_percentile is None:
            cap_percentile = self.response_cap_percentile

        output = np.zeros_like(
            mask,
            dtype=np.uint8
        )

        pixels = response[mask > 0]

        if pixels.size == 0:
            return output

        floor_value = np.percentile(
            pixels,
            floor_percentile
        )

        cap_value = np.percentile(
            pixels,
            cap_percentile
        )

        if cap_value <= floor_value:
            return output

        temp = response.astype(
            np.float32
        )

        temp = (temp - floor_value) / (cap_value - floor_value)

        temp = np.clip(
            temp,
            0.0,
            1.0
        )

        temp = (temp * 255).astype(
            np.uint8
        )

        temp = cv2.bitwise_and(
            temp,
            temp,
            mask=mask
        )

        return temp

    def compute_frangi_response(self, green, fundus_mask):
        green_norm = self.normalize_inside_mask(
            green,
            fundus_mask
        )

        img = green_norm.astype(
            np.float32
        )

        vesselness_max = np.zeros_like(
            img,
            dtype=np.float32
        )

        mask_bool = fundus_mask > 0

        for sigma in self.frangi_sigmas:
            ksize = int(6 * sigma + 1)

            if ksize % 2 == 0:
                ksize += 1

            smoothed = cv2.GaussianBlur(
                img,
                (ksize, ksize),
                sigma
            )

            ixx = cv2.Sobel(
                smoothed,
                cv2.CV_32F,
                2,
                0,
                ksize=3
            ) * (sigma ** 2)

            iyy = cv2.Sobel(
                smoothed,
                cv2.CV_32F,
                0,
                2,
                ksize=3
            ) * (sigma ** 2)

            ixy = cv2.Sobel(
                smoothed,
                cv2.CV_32F,
                1,
                1,
                ksize=3
            ) * (sigma ** 2)

            tmp = np.sqrt(
                np.maximum(
                    (ixx - iyy) ** 2 + 4.0 * (ixy ** 2),
                    0.0
                )
            )

            lambda1 = 0.5 * (ixx + iyy + tmp)
            lambda2 = 0.5 * (ixx + iyy - tmp)

            swap = np.abs(lambda1) > np.abs(lambda2)

            l1 = np.where(
                swap,
                lambda2,
                lambda1
            )

            l2 = np.where(
                swap,
                lambda1,
                lambda2
            )

            l2_safe = np.where(
                np.abs(l2) < 1e-6,
                1e-6,
                l2
            )

            rb = np.abs(l1) / np.abs(l2_safe)

            s = np.sqrt(
                l1 ** 2 + l2 ** 2
            )

            if np.any(mask_bool):
                s_max = float(
                    np.max(s[mask_bool])
                )
            else:
                s_max = 0.0

            gamma = max(
                self.frangi_gamma_min,
                0.5 * s_max
            )

            vesselness = np.exp(
                -(rb ** 2) / (2.0 * self.frangi_beta ** 2)
            )

            vesselness = vesselness * (
                1.0 - np.exp(
                    -(s ** 2) / (2.0 * gamma ** 2)
                )
            )

            vesselness = np.where(
                l2 > 0,
                vesselness,
                0.0
            )

            vesselness_max = np.maximum(
                vesselness_max,
                vesselness
            )

        vesselness_max = np.where(
            mask_bool,
            vesselness_max,
            0.0
        )

        frangi_response = self.normalize_response_with_floor(
            vesselness_max,
            fundus_mask,
            floor_percentile=self.response_floor_percentile,
            cap_percentile=self.response_cap_percentile
        )

        return frangi_response, green_norm

    def compute_gabor_response(self, green_norm, fundus_mask):
        inverted = 255 - green_norm

        inverted = cv2.bitwise_and(
            inverted,
            inverted,
            mask=fundus_mask
        )

        img = inverted.astype(
            np.float32
        )

        response_max = np.zeros_like(
            img,
            dtype=np.float32
        )

        for kernel in self.gabor_kernels:
            response = cv2.filter2D(
                img,
                cv2.CV_32F,
                kernel
            )

            response = np.maximum(
                response,
                0.0
            )

            response_max = np.maximum(
                response_max,
                response
            )

        gabor_response = self.normalize_response_with_floor(
            response_max,
            fundus_mask,
            floor_percentile=self.response_floor_percentile,
            cap_percentile=self.response_cap_percentile
        )

        return gabor_response

    def combine_frangi_gabor(self, frangi_response, gabor_response, fundus_mask):
        """
        FrangiとGaborを統合する。

        Frangi x Gabor の積だけだと薄い血管や曲がった血管が落ちやすい。
        そのため、積を主成分にしつつ、weighted responseを一部残す。
        """

        frangi_f = frangi_response.astype(np.float32) / 255.0
        gabor_f = gabor_response.astype(np.float32) / 255.0

        product_response = np.sqrt(
            np.maximum(
                frangi_f * gabor_f,
                0.0
            )
        )

        weighted_response = (
            self.frangi_weight * frangi_f +
            self.gabor_weight * gabor_f
        )

        combined = np.maximum(
            product_response,
            0.45 * weighted_response
        )

        combined = np.clip(
            combined,
            0.0,
            1.0
        )

        combined = (combined * 255).astype(np.uint8)

        combined = cv2.bitwise_and(
            combined,
            combined,
            mask=fundus_mask
        )

        if self.response_median_ksize >= 3:
            ksize = self.response_median_ksize

            if ksize % 2 == 0:
                ksize += 1

            combined = cv2.medianBlur(
                combined,
                ksize
            )

        combined = cv2.GaussianBlur(
            combined,
            (3, 3),
            0
        )

        combined = self.normalize_response_with_floor(
            combined.astype(np.float32),
            fundus_mask,
            floor_percentile=max(55, self.response_floor_percentile - 10),
            cap_percentile=self.response_cap_percentile
        )

        return combined

    def extract_vessel_response(self, green, fundus_mask):
        frangi_response, green_norm = self.compute_frangi_response(
            green,
            fundus_mask
        )

        gabor_response = self.compute_gabor_response(
            green_norm,
            fundus_mask
        )

        combined_response = self.combine_frangi_gabor(
            frangi_response,
            gabor_response,
            fundus_mask
        )

        return combined_response, frangi_response, gabor_response, green_norm

    def hysteresis_threshold(self, response, fundus_mask, frangi_response=None):
        """
        strong seedはcombined responseから作る。
        weak candidateはcombined response + Frangi-only responseから作る。

        これにより、Gaborでは落ちやすい薄い血管も、
        strong seedに接続していれば救済できる。
        """

        valid = (fundus_mask > 0) & (response >= self.min_valid_response)

        pixels = response[valid]

        if pixels.size == 0:
            empty = np.zeros_like(response, dtype=np.uint8)
            return empty, empty, 0.0, 0.0

        high_threshold = np.percentile(
            pixels,
            self.high_percentile
        )

        low_threshold = high_threshold * self.low_ratio

        strong = np.zeros_like(response, dtype=np.uint8)
        weak = np.zeros_like(response, dtype=np.uint8)

        strong[
            (response >= high_threshold) & (fundus_mask > 0)
        ] = 255

        weak[
            (response >= low_threshold) & (fundus_mask > 0)
        ] = 255

        if frangi_response is not None:
            frangi_valid = (
                (fundus_mask > 0) &
                (frangi_response >= self.weak_frangi_min_response)
            )

            frangi_pixels = frangi_response[frangi_valid]

            if frangi_pixels.size > 0:
                frangi_high = np.percentile(
                    frangi_pixels,
                    self.high_percentile
                )

                frangi_weak_threshold = frangi_high * self.weak_frangi_ratio

                frangi_weak = np.zeros_like(response, dtype=np.uint8)

                frangi_weak[
                    (frangi_response >= frangi_weak_threshold) &
                    (fundus_mask > 0)
                ] = 255

                weak = cv2.bitwise_or(
                    weak,
                    frangi_weak
                )

        if self.gap_close_size >= 3:
            close_size = self.gap_close_size

            if close_size % 2 == 0:
                close_size += 1

            close_kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE,
                (close_size, close_size)
            )

            weak = cv2.morphologyEx(
                weak,
                cv2.MORPH_CLOSE,
                close_kernel
            )

            weak = cv2.bitwise_and(
                weak,
                weak,
                mask=fundus_mask
            )

        connection_size = self.weak_connection_dilate_size

        if connection_size % 2 == 0:
            connection_size += 1

        connection_kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (
                connection_size,
                connection_size
            )
        )

        strong_neighborhood = cv2.dilate(
            strong,
            connection_kernel,
            iterations=1
        )

        num_labels, labels = cv2.connectedComponents(
            weak,
            connectivity=8
        )

        result = np.zeros_like(response, dtype=np.uint8)

        for label_id in range(1, num_labels):
            component = labels == label_id

            touches_strong = np.any(
                strong_neighborhood[component] > 0
            )

            if touches_strong:
                result[component] = 255

        return result, strong, float(high_threshold), float(low_threshold)

    def morphological_skeleton(self, binary):
        binary = (binary > 0).astype(
            np.uint8
        ) * 255

        if not self.use_skeleton:
            return binary

        if hasattr(cv2, "ximgproc"):
            skeleton = cv2.ximgproc.thinning(
                binary,
                thinningType=cv2.ximgproc.THINNING_ZHANGSUEN
            )

            return skeleton

        skeleton = np.zeros_like(
            binary
        )

        element = cv2.getStructuringElement(
            cv2.MORPH_CROSS,
            (3, 3)
        )

        temp = binary.copy()

        while True:
            opened = cv2.morphologyEx(
                temp,
                cv2.MORPH_OPEN,
                element
            )

            subtracted = cv2.subtract(
                temp,
                opened
            )

            eroded = cv2.erode(
                temp,
                element
            )

            skeleton = cv2.bitwise_or(
                skeleton,
                subtracted
            )

            temp = eroded.copy()

            if cv2.countNonZero(temp) == 0:
                break

        return skeleton

    def remove_components_by_linearity(self, binary, response, strong_seed=None):
        """
        Connected Componentごとに、
        面積、長さ、応答、BBox形状、Skeleton length / Area を使って判定する。
        """

        binary = (binary > 0).astype(
            np.uint8
        ) * 255

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            binary,
            connectivity=8
        )

        cleaned = np.zeros_like(
            binary
        )

        h, w = binary.shape

        if strong_seed is not None:
            dilate_kernel = np.ones(
                (
                    self.strong_connection_dilate_size,
                    self.strong_connection_dilate_size
                ),
                np.uint8
            )

            strong_dilated = cv2.dilate(
                strong_seed,
                dilate_kernel,
                iterations=1
            )
        else:
            strong_dilated = None

        for label_id in range(1, num_labels):
            area = stats[label_id, cv2.CC_STAT_AREA]
            x = stats[label_id, cv2.CC_STAT_LEFT]
            y = stats[label_id, cv2.CC_STAT_TOP]
            width = stats[label_id, cv2.CC_STAT_WIDTH]
            height = stats[label_id, cv2.CC_STAT_HEIGHT]

            bbox_length = max(
                width,
                height
            )

            bbox_short = max(
                1,
                min(width, height)
            )

            aspect_ratio = bbox_length / bbox_short

            bbox_area = max(
                1,
                width * height
            )

            extent = area / bbox_area

            component_mask = labels == label_id

            if self.remove_image_border_components:
                touches_image_border = (
                    x <= self.image_border_margin or
                    y <= self.image_border_margin or
                    x + width >= w - self.image_border_margin or
                    y + height >= h - self.image_border_margin
                )

                if touches_image_border:
                    continue

            connected_to_strong = False

            if strong_dilated is not None:
                connected_to_strong = bool(
                    np.any(strong_dilated[component_mask] > 0)
                )

            if connected_to_strong:
                min_area = max(
                    1,
                    int(self.min_component_area * self.connected_area_ratio)
                )

                min_length = max(
                    1,
                    int(self.min_component_length * self.connected_length_ratio)
                )

                min_response = float(
                    self.min_response_mean * self.connected_response_ratio
                )

                min_linearity = float(
                    self.min_linearity * self.connected_min_linearity_ratio
                )
            else:
                min_area = self.min_component_area
                min_length = self.min_component_length
                min_response = self.min_response_mean
                min_linearity = self.min_linearity

            if area < min_area:
                continue

            if bbox_length < min_length:
                continue

            component_response_mean = float(
                np.mean(
                    response[component_mask]
                )
            )

            if component_response_mean < min_response:
                continue

            compact_blob = (
                aspect_ratio < self.min_aspect_ratio and
                area < self.min_component_area * 3
            )

            if compact_blob:
                continue

            too_filled_blob = (
                extent > self.max_component_extent and
                area < self.min_component_area * 4
            )

            if too_filled_blob:
                continue

            component_image = np.zeros_like(
                binary
            )

            component_image[component_mask] = 255

            component_skeleton = self.morphological_skeleton(
                component_image
            )

            skeleton_length = np.count_nonzero(
                component_skeleton
            )

            linearity = skeleton_length / max(
                float(area),
                1.0
            )

            if linearity < min_linearity:
                continue

            cleaned[component_mask] = 255

        return cleaned

    def vessel_distribution_score(self, vessel_binary):
        binary = vessel_binary > 0

        h, w = binary.shape

        cell_h = h // self.grid_size
        cell_w = w // self.grid_size

        if cell_h == 0 or cell_w == 0:
            return 0.0

        occupied_cells = 0
        total_cells = self.grid_size * self.grid_size

        for gy in range(self.grid_size):
            for gx in range(self.grid_size):
                y0 = gy * cell_h
                y1 = h if gy == self.grid_size - 1 else (gy + 1) * cell_h

                x0 = gx * cell_w
                x1 = w if gx == self.grid_size - 1 else (gx + 1) * cell_w

                cell = binary[y0:y1, x0:x1]

                if np.any(cell):
                    occupied_cells += 1

        score = occupied_cells / total_cells

        return score

    def calculate_vessel_metrics(self, vessel_mask, vessel_skeleton, fundus_mask):
        fundus_area = np.count_nonzero(
            fundus_mask
        )

        if fundus_area == 0:
            return {
                "vessel_area_ratio": 0.0,
                "skeleton_length_ratio": 0.0,
                "distribution_score": 0.0,
            }

        vessel_area = np.count_nonzero(
            vessel_mask
        )

        skeleton_length = np.count_nonzero(
            vessel_skeleton
        )

        return {
            "vessel_area_ratio": float(vessel_area / fundus_area),
            "skeleton_length_ratio": float(skeleton_length / fundus_area),
            "distribution_score": float(
                self.vessel_distribution_score(vessel_skeleton)
            ),
        }

    def extract_one_image(self, green_original, fundus_mask_original):
        original_shape = green_original.shape

        green, fundus_mask = self.resize_for_processing(
            green_original,
            fundus_mask_original
        )

        steps = {}

        effective_mask = self.create_effective_mask(
            fundus_mask
        )

        (
            vessel_response,
            frangi_response,
            gabor_response,
            green_norm
        ) = self.extract_vessel_response(
            green,
            effective_mask
        )

        vessel_mask_raw, strong_seed, high_threshold, low_threshold = self.hysteresis_threshold(
            vessel_response,
            effective_mask,
            frangi_response=frangi_response
        )

        vessel_mask_cleaned = self.remove_components_by_linearity(
            vessel_mask_raw,
            vessel_response,
            strong_seed=strong_seed
        )

        vessel_mask_cleaned = cv2.bitwise_and(
            vessel_mask_cleaned,
            vessel_mask_cleaned,
            mask=effective_mask
        )

        vessel_skeleton = self.morphological_skeleton(
            vessel_mask_cleaned
        )

        vessel_mask_original = self.resize_result_to_original(
            vessel_mask_cleaned,
            original_shape
        )

        vessel_skeleton_original = self.resize_result_to_original(
            vessel_skeleton,
            original_shape
        )

        effective_mask_original = self.resize_result_to_original(
            effective_mask,
            original_shape
        )

        metrics = self.calculate_vessel_metrics(
            vessel_mask_original,
            vessel_skeleton_original,
            effective_mask_original
        )

        metrics["process_scale"] = self.process_scale
        metrics["high_threshold"] = high_threshold
        metrics["low_threshold"] = low_threshold

        if self.save_debug:
            steps["01_green_input"] = green.copy()
            steps["02_fundus_mask_effective"] = effective_mask.copy()
            steps["03_green_normalized"] = green_norm.copy()
            steps["04_frangi_response"] = frangi_response.copy()
            steps["05_gabor_response"] = gabor_response.copy()
            steps["06_combined_response"] = vessel_response.copy()
            steps["07_strong_seed"] = strong_seed.copy()
            steps["08_vessel_mask_raw"] = vessel_mask_raw.copy()
            steps["09_vessel_mask_cleaned"] = vessel_mask_cleaned.copy()
            steps["10_vessel_skeleton"] = vessel_skeleton.copy()

        return vessel_mask_original, vessel_skeleton_original, steps, metrics

    def save_debug_steps(self, steps, save_dir):
        os.makedirs(
            save_dir,
            exist_ok=True
        )

        for name, image in steps.items():
            save_path = os.path.join(
                save_dir,
                name + ".png"
            )

            cv2.imwrite(
                save_path,
                image
            )

    def run(self):
        files = sorted([
            file for file in os.listdir(self.input_green_dir)
            if file.lower().endswith(".png")
        ])

        if len(files) == 0:
            raise FileNotFoundError(
                f"{self.input_green_dir} にpng画像がありません。"
            )

        random.seed(
            self.random_seed
        )

        debug_files = set(
            random.sample(
                files,
                min(
                    self.debug_sample_count,
                    len(files)
                )
            )
        )

        records = []
        debug_index = 1

        for file in tqdm(files):
            image_path = os.path.join(
                self.input_green_dir,
                file
            )

            green = cv2.imread(
                image_path,
                cv2.IMREAD_GRAYSCALE
            )

            if green is None:
                print(f"画像を読み込めませんでした: {image_path}")
                continue

            fundus_mask, fundus_mask_path, mask_source = self.load_fundus_mask(
                file,
                green
            )

            vessel_mask, vessel_skeleton, steps, metrics = self.extract_one_image(
                green,
                fundus_mask
            )

            base_id, image_id = self.get_base_image_id(
                file
            )

            mask_output_path = os.path.join(
                self.output_mask_dir,
                image_id + "_vessel_mask.png"
            )

            skeleton_output_path = os.path.join(
                self.output_skeleton_dir,
                image_id + "_vessel_skeleton.png"
            )

            cv2.imwrite(
                mask_output_path,
                vessel_mask
            )

            cv2.imwrite(
                skeleton_output_path,
                vessel_skeleton
            )

            records.append({
                "image": file,
                "base_id": base_id,
                "fundus_mask_path": fundus_mask_path,
                "mask_source": mask_source,
                "vessel_mask_output": mask_output_path,
                "skeleton_output": skeleton_output_path,
                "vessel_area_ratio": metrics["vessel_area_ratio"],
                "skeleton_length_ratio": metrics["skeleton_length_ratio"],
                "distribution_score": metrics["distribution_score"],
                "process_scale": metrics["process_scale"],
                "high_threshold": metrics["high_threshold"],
                "low_threshold": metrics["low_threshold"],
                "effective_mask_erode_size": self.effective_mask_erode_size,
                "frangi_sigmas": str(self.frangi_sigmas),
                "frangi_beta": self.frangi_beta,
                "frangi_gamma_min": self.frangi_gamma_min,
                "gabor_ksize": self.gabor_ksize,
                "gabor_sigmas": str(self.gabor_sigmas),
                "gabor_lambdas": str(self.gabor_lambdas),
                "gabor_gamma": self.gabor_gamma,
                "gabor_angle_step": self.gabor_angle_step,
                "frangi_weight": self.frangi_weight,
                "gabor_weight": self.gabor_weight,
                "use_product_response": self.use_product_response,
                "response_floor_percentile": self.response_floor_percentile,
                "response_cap_percentile": self.response_cap_percentile,
                "high_percentile": self.high_percentile,
                "low_ratio": self.low_ratio,
                "min_valid_response": self.min_valid_response,
                "gap_close_size": self.gap_close_size,
                "weak_frangi_ratio": self.weak_frangi_ratio,
                "weak_frangi_min_response": self.weak_frangi_min_response,
                "weak_connection_dilate_size": self.weak_connection_dilate_size,
                "min_component_area": self.min_component_area,
                "min_component_length": self.min_component_length,
                "min_response_mean": self.min_response_mean,
                "min_aspect_ratio": self.min_aspect_ratio,
                "max_component_extent": self.max_component_extent,
                "min_linearity": self.min_linearity,
                "connected_area_ratio": self.connected_area_ratio,
                "connected_length_ratio": self.connected_length_ratio,
                "connected_response_ratio": self.connected_response_ratio,
                "connected_min_linearity_ratio": self.connected_min_linearity_ratio,
                "strong_connection_dilate_size": self.strong_connection_dilate_size,
                "use_skeleton": self.use_skeleton,
            })

            if self.save_debug and file in debug_files:
                sample_dir = os.path.join(
                    self.debug_dir,
                    f"sample{debug_index:02d}_{image_id}"
                )

                self.save_debug_steps(
                    steps,
                    sample_dir
                )

                debug_index += 1

        df = pd.DataFrame(
            records
        )

        df.to_csv(
            self.metrics_csv,
            index=False
        )

        print("血管抽出が完了しました。")
        print(f"入力G画像: {self.input_green_dir}")
        print(f"入力眼底領域マスク: {self.input_mask_dir}")
        print(f"血管マスク画像: {self.output_mask_dir}")
        print(f"血管スケルトン画像: {self.output_skeleton_dir}")
        print(f"デバッグ画像: {self.debug_dir}")
        print(f"血管抽出指標CSV: {self.metrics_csv}")


if __name__ == "__main__":
    extractor = FundusVesselExtractor(
        input_green_dir="preprocessed_green_images",
        input_mask_dir="fundus_masks",
        output_mask_dir="vessel_masks",
        output_skeleton_dir="vessel_skeletons",
        debug_dir="vessel_debug_samples",
        metrics_csv="vessel_extraction_metrics.csv",

        debug_sample_count=1,
        random_seed=42,
        grid_size=8,

        process_scale=0.85,

        effective_mask_erode_size=20,

        frangi_sigmas=(0.8, 1.0, 1.4, 2.0, 2.8),
        frangi_beta=0.35,
        frangi_gamma_min=2.0,

        gabor_ksize=17,
        gabor_sigmas=(2.0, 3.0),
        gabor_lambdas=(8.0, 12.0),
        gabor_gamma=0.45,
        gabor_angle_step=30,

        frangi_weight=0.65,
        gabor_weight=0.35,
        use_product_response=True,

        response_floor_percentile=62,
        response_cap_percentile=99.5,
        response_median_ksize=3,

        high_percentile=87.0,
        low_ratio=0.38,
        min_valid_response=5,
        gap_close_size=5,

        weak_frangi_ratio=0.50,
        weak_frangi_min_response=4,
        weak_connection_dilate_size=11,

        min_component_area=20,
        min_component_length=10,
        min_response_mean=5.0,
        min_aspect_ratio=1.4,
        max_component_extent=0.92,

        min_linearity=0.12,
        connected_min_linearity_ratio=0.55,

        connected_area_ratio=0.45,
        connected_length_ratio=0.45,
        connected_response_ratio=0.55,
        strong_connection_dilate_size=7,

        remove_image_border_components=True,
        image_border_margin=4,

        use_skeleton=True,
        save_debug=True,
    )

    extractor.run()

100%|██████████| 1434/1434 [1:08:10<00:00,  2.85s/it]


血管抽出が完了しました。
入力G画像: preprocessed_green_images
入力眼底領域マスク: fundus_masks
血管マスク画像: vessel_masks
血管スケルトン画像: vessel_skeletons
デバッグ画像: vessel_debug_samples
血管抽出指標CSV: vessel_extraction_metrics.csv
